In [ ]:
import pandas as pd

In [ ]:
# Manual fasta parser
def manual_fasta_parser(file_path):
    #makes an empty dict
    sequences = {}
    #says i aint pointing at anything
    current_header = None
    # Open the target file using a standard context manager
    with open(file_path, 'r') as f:
        for line in f:
            # 1. Clean up the trailing whitespace or newline blocks (ya know \n)
            line = line.strip()
            # 2. Check if the line is empty (skip it if it is)
            if not line:
                continue
            # 3. Check if the line represents an identity label
            if line.startswith('>'):
                current_header = line[1:]
                #makes the thing like xyz_1234 and not >xyz_1234, and makes it the key with an empty value
                sequences[current_header] = ''
                
            # 4. If it's not a label, it MUST be sequence data
            else:
                #that is we have a label, and thus the key gets assigned the string as the value
                if current_header is not None:
                    sequences[current_header] += line
                
    return sequences

In [ ]:
def calculate_highest_gc_content(sequences):
    """
    Accepts a dictionary of FASTA sequences, calculates the GC percentage 
    for each, and returns the label and value of the highest GC content.
    """
    GCPercent = {}

    for label, seq in sequences.items():
        # Calculates GC percent in a given sequence
        g_count = seq.count('G')
        c_count = seq.count('C')
        seq_length = len(seq)
        
        if seq_length > 0:
            GC_content = (g_count + c_count) / seq_length * 100
            GCPercent[label] = GC_content
        else:
            GCPercent[label] = 0.0
            
    # Convert to a pandas Series to leverage optimized maximum indexing
    gc = pd.Series(GCPercent)
    max_key = gc.idxmax()
    max_val = gc.get(max_key)
    
    return max_key, max_val

In [ ]:
# ==========================================
# 1. CODON DICTIONARY (RNA COMPLIANT)
# ==========================================
codon_table = {
    'UUU':'F','UUC': 'F', 'UUA': 'L', 'UUG': 'L',
    'UCU': 'S', 'UCC': 'S', 'UCA': 'S', 'UCG': 'S',
    'UAU': 'Y', 'UAC': 'Y', 'UAA': 'Stop', 'UAG': 'Stop',
    'UGU': 'C', 'UGC': 'C', 'UGA': 'Stop', 'UGG': 'W',
    'CUU': 'L', 'CUC': 'L', 'CUA': 'L', 'CUG': 'L',
    'CCU': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAU': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGU': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AUU': 'I', 'AUC': 'I', 'AUA': 'I', 'AUG': 'M',
    'ACU': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAU': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGU': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GUU': 'V', 'GUC': 'V', 'GUA': 'V', 'GUG': 'V',
    'GCU': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAU': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGU': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

# ==========================================
# 2. CORE TRANSLATION FUNCTION (STEP-3 SLICE)
# ==========================================
def translate_orf_slice(orf_slice):
    protein = ""
    # Walk down the slice in increments of 3
    for j in range(0, len(orf_slice), 3):
        codon = orf_slice[j:j+3]
        
        # If the trailing chunk is less than 3 letters, we reached the end 
        # without a stop codon. This is an invalid frame -> discard it.
        if len(codon) < 3:
            return None
            
        amino_acid = codon_table.get(codon)
        
        # Stop codon encountered -> valid protein string complete!
        if amino_acid == 'Stop':
            return protein
            
        # Missing codon or dictionary error -> discard the frame
        if not amino_acid:
            return None
            
        protein += amino_acid
        
    # If the loop runs out of text completely without hitting a Stop codon -> discard it
    return None

# ==========================================
# 3. MASTER PROCESSING ENGINES
# ==========================================
existing_fasta_data = manual_fasta_parser('rosalind_orf.txt')


# Master set to collect all unique proteins for the final Rosalind output matrix
final_unique_proteins = set()

# Loop through your existing reader dictionary using your explicit lookup logic
for label in existing_fasta_data:
    dna_sequence = existing_fasta_data[label]
    
    # Track A: Forward RNA Transcription
    rna_forward = dna_sequence.upper().replace('T', 'U')
    
    # Track B: Reverse Complement Transcription (Simultaneous Base Swapping)
    rna_pairs = str.maketrans('AUGC', 'UACG')
    rna_reverse = rna_forward.translate(rna_pairs)[::-1]
    
    # Put both tracks in a list to process them cleanly using the exact same loops
    tracks = [rna_forward, rna_reverse]
    
    for rna_track in tracks:
        # Extract every index where an 'AUG' start codon appears in this track
        start_indices = []
        for i in range(len(rna_track) - 2):
            if rna_track[i:i+3] == "AUG":
                start_indices.append(i)
                
        # Loop through your found start indices and feed the slices to the translator
        for start_pos in start_indices:
            # Slice from the discovered start index to the absolute end of the track
            current_slice = rna_track[start_pos:]
            
            # Pass the slice to the rigid translation gate
            protein_sequence = translate_orf_slice(current_slice)
            
            # If the frame contained a stop codon and returned text, store it
            if protein_sequence is not None:
                final_unique_proteins.add(protein_sequence)

# ==========================================
# 4. OUTPUT EXECUTION
# ==========================================
# Print each distinct protein sequence on a fresh line for the Rosalind upload
for protein in final_unique_proteins:
    print(protein)
